### Setup

In [4]:
from pathlib import Path
import os
import sys
sys.path.append(os.path.abspath(".."))
from utils.data import load_mc_pacman_data, print_data_structure
import pickle

DATA = Path(os.environ["DATA"])
DATA_PATH = DATA / "mc_pacman.pkl"
TMP_DIR = DATA / "tmp"
TMP_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("TMPDIR", str(TMP_DIR))
os.environ.setdefault("MPLCONFIGDIR", str(TMP_DIR / "matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

MODEL_DIR = Path(os.environ.get("POISSON_LDS_MODEL_DIR", "models"))
MODEL_DIR.mkdir(parents=True, exist_ok=True)

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from plotly.subplots import make_subplots
from jax import vmap
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import r2_score

from dynamax.generalized_gaussian_ssm import GeneralizedGaussianSSM, EKFIntegrals
from dynamax.generalized_gaussian_ssm import conditional_moments_gaussian_smoother as cmgs

BIN_MS = 50
SMOOTH_SIGMA_BINS = 1.0
STATE_DIM = 18
LEARNING_STEPS = 100
LEARNING_RATE = 1e-2
TEST_FRACTION = 0.2
SEED = 7
RIDGE_ALPHAS = np.logspace(-3, 3, 13)


In [5]:
from utils.preprocessing import (
    bin_and_smooth_trials, 
    split_condition_trials, 
    pooled_split_indices,
    standardize_train_test, 
    stack_trials
)

from utils.poisson_lds import (
    COV_FLOOR,
    RATE_FLOOR,
    PoissonLDSParams,
    infer_latents,
    infer_latents_grouped,
    learn_poisson_lds,
    learn_poisson_lds_grouped,
    make_poisson_lds_params,
    marginal_log_prob_per_bin,
    poisson_rate,
    predict_expected_counts,
)

from utils.lds_cache import (
    lds_metadata as make_lds_metadata,
    lds_path as make_lds_path,
    load_or_fit_lds,
    poisson_fit_payload_for_pickle,
    restore_poisson_fit_from_pickle,
)

In [6]:
data = load_mc_pacman_data()
conditions = np.asarray(data["condition"])
force_trials = data["force"]
spike_trials = data["spikes"]


### Model fitting functions

In [7]:
def fit_condition_lds(
    cond,
    learning_steps=LEARNING_STEPS,
    state_dim=STATE_DIM,
    seed=SEED,
    bin_ms=BIN_MS,
    smooth_sigma_bins=SMOOTH_SIGMA_BINS,
    learning_rate=LEARNING_RATE,
    test_fraction=TEST_FRACTION,
    verbose=False,
):
    train_idx, test_idx = split_condition_trials(conditions, cond, test_fraction, seed)
    train_spikes = [spike_trials[i] for i in train_idx]
    test_spikes = [spike_trials[i] for i in test_idx]
    train_force = [force_trials[i] for i in train_idx]
    test_force = [force_trials[i] for i in test_idx]

    y_train = bin_and_smooth_trials(train_spikes, bin_ms, 0, "spikes")
    y_test = bin_and_smooth_trials(test_spikes, bin_ms, 0, "spikes")
    f_train = bin_and_smooth_trials(train_force, bin_ms, smooth_sigma_bins, "force")
    f_test = bin_and_smooth_trials(test_force, bin_ms, smooth_sigma_bins, "force")

    y_train_stack = stack_trials(y_train)
    model = GeneralizedGaussianSSM(state_dim=state_dim, emission_dim=y_train_stack.shape[-1])
    key = jr.PRNGKey(seed + 1000 + int(cond))
    trainable_params, params, lls = learn_poisson_lds(
        y_train_stack,
        state_dim=state_dim,
        key=key,
        learning_steps=learning_steps,
        learning_rate=learning_rate,
        verbose=verbose,
    )

    return {
        "condition": int(cond),
        "model": model,
        "params": params,
        "trainable_params": trainable_params,
        "training_log_likelihoods": lls,
        "train_idx": train_idx,
        "test_idx": test_idx,
        "y_train": y_train,
        "y_test": y_test,
        "force_train": f_train,
        "force_test": f_test,
    }


def fit_pooled_lds(
    learning_steps=LEARNING_STEPS,
    state_dim=STATE_DIM,
    seed=SEED,
    bin_ms=BIN_MS,
    smooth_sigma_bins=SMOOTH_SIGMA_BINS,
    learning_rate=LEARNING_RATE,
    test_fraction=TEST_FRACTION,
    verbose=False,
):
    train_idx, test_idx = pooled_split_indices(conditions, test_fraction, seed)
    train_spikes = [spike_trials[i] for i in train_idx]
    test_spikes = [spike_trials[i] for i in test_idx]
    train_force = [force_trials[i] for i in train_idx]
    test_force = [force_trials[i] for i in test_idx]

    y_train = bin_and_smooth_trials(train_spikes, bin_ms, 0, "spikes")
    y_test = bin_and_smooth_trials(test_spikes, bin_ms, 0, "spikes")
    f_train = bin_and_smooth_trials(train_force, bin_ms, smooth_sigma_bins, "force")
    f_test = bin_and_smooth_trials(test_force, bin_ms, smooth_sigma_bins, "force")

    model = GeneralizedGaussianSSM(state_dim=state_dim, emission_dim=y_train[0].shape[-1])
    key = jr.PRNGKey(seed + 2000)
    trainable_params, params, lls = learn_poisson_lds_grouped(
        y_train,
        state_dim=state_dim,
        key=key,
        learning_steps=learning_steps,
        learning_rate=learning_rate,
        verbose=verbose,
    )

    return {
        "condition": "pooled",
        "model": model,
        "params": params,
        "trainable_params": trainable_params,
        "training_log_likelihoods_per_bin": lls,
        "train_idx": train_idx,
        "test_idx": test_idx,
        "condition_train": np.asarray(conditions[train_idx], dtype=int),
        "condition_test": np.asarray(conditions[test_idx], dtype=int),
        "y_train": y_train,
        "y_test": y_test,
        "force_train": f_train,
        "force_test": f_test,
    }


predict_rates = predict_expected_counts


In [8]:
def lds_metadata(
    kind,
    *,
    learning_steps,
    state_dim,
    seed,
    bin_ms,
    smooth_sigma_bins,
    learning_rate,
    rate_floor,
    cov_floor,
    test_fraction,
    data_path,
    emission_distribution,
    condition=None,
):
    emission_distribution = str(emission_distribution).lower()
    if emission_distribution != "poisson":
        raise ValueError(f"unsupported emission distribution: {emission_distribution}")
    return make_lds_metadata(
        MODEL_CACHE_VERSION,
        kind,
        emission_distribution,
        data_path,
        condition=condition,
        bin_ms=int(bin_ms),
        smooth_sigma_bins=float(smooth_sigma_bins),
        state_dim=int(state_dim),
        learning_steps=int(learning_steps),
        learning_rate=float(learning_rate),
        rate_floor=float(rate_floor),
        cov_floor=float(cov_floor),
        test_fraction=float(test_fraction),
        seed=int(seed),
    )


def lds_path(kind, metadata):
    return make_lds_path(MODEL_DIR, kind, metadata)


def load_or_fit_condition_lds(
    cond,
    learning_steps=LEARNING_STEPS,
    state_dim=STATE_DIM,
    seed=SEED,
    bin_ms=BIN_MS,
    smooth_sigma_bins=SMOOTH_SIGMA_BINS,
    learning_rate=LEARNING_RATE,
    rate_floor=RATE_FLOOR,
    cov_floor=COV_FLOOR,
    test_fraction=TEST_FRACTION,
    data_path=DATA_PATH,
    emission_distribution="poisson",
    verbose=False,
):
    metadata = lds_metadata(
        "condition_lds",
        learning_steps=learning_steps,
        state_dim=state_dim,
        seed=seed,
        bin_ms=bin_ms,
        smooth_sigma_bins=smooth_sigma_bins,
        learning_rate=learning_rate,
        rate_floor=rate_floor,
        cov_floor=cov_floor,
        test_fraction=test_fraction,
        data_path=data_path,
        emission_distribution=emission_distribution,
        condition=cond,
    )
    path = lds_path("condition_lds", metadata)
    return load_or_fit_lds(
        path,
        metadata,
        lambda: fit_condition_lds(
            cond=int(cond),
            learning_steps=learning_steps,
            state_dim=state_dim,
            seed=seed,
            bin_ms=bin_ms,
            smooth_sigma_bins=smooth_sigma_bins,
            learning_rate=learning_rate,
            test_fraction=test_fraction,
            verbose=verbose,
        ),
        poisson_fit_payload_for_pickle,
        restore_poisson_fit_from_pickle,
    )
